# ODI to Databricks Migration

**Source File:** `TARGET.sql`

**Conversion Timestamp:** 2024-07-30T12:00:00Z

**Description:** This notebook converts an ODI SQL script for inserting into `TRG_LOC` table into Databricks Spark SQL.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "", "ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")

## ETL Parameters

In [ ]:
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
    '${ETL_PROC_WID}' AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO
"""))

## Load Target Table (SCEN_TASK_NO {30})

In [ ]:
%sql
-- SCEN_TASK_NO in {10}: Initial step, no SQL operation.
-- SCEN_TASK_NO in {20}: Placeholder step, no SQL operation.
INSERT INTO workspace.hr.trg_loc
  (
    location_id,
    street_address,
    postal_code,
    city,
    state_province,
    country_id
  )
SELECT
  locations.location_id,
  locations.street_address,
  locations.postal_code,
  locations.city,
  locations.state_province,
  locations.country_id
FROM
  workspace.hr.locations AS locations;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS trg_loc_record_count FROM workspace.hr.trg_loc;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Naming:** Oracle schema `HR` has been mapped to `workspace.hr`. Table names `TRG_LOC` and `LOCATIONS` have been lowercased to `trg_loc` and `locations` respectively, and fully qualified with `workspace.hr`.
2.  **Oracle Hints:** The Oracle `/*+ APPEND PARALLEL */` hint has been removed as it is not applicable to Databricks Delta Lake.
3.  **Data Types:** Implicit data type mapping will occur. Ensure `LOCATION_ID` and `COUNTRY_ID` in `workspace.hr.trg_loc` are of appropriate types (e.g., `BIGINT` or `STRING`), and string columns (`STREET_ADDRESS`, `POSTAL_CODE`, `CITY`, `STATE_PROVINCE`) are `STRING`.
4.  **Initial Table Existence:** This script assumes `workspace.hr.trg_loc` and `workspace.hr.locations` tables already exist with compatible schemas. If `trg_loc` does not exist, a `CREATE TABLE` statement would be required before this `INSERT`.
